# Species Identification with Perch V2 

This notebook runs a complete species-identification pipeline using the
Perch V2 model downloaded from Kaggle.

## Workflow

1. **Download** Perch V2 from Kaggle with `kagglehub` (cached after the first run).
2. **Load** the TensorFlow SavedModel and its bundled list of species labels.
3. For each recording:
   - **load & resample** the audio to mono 32 kHz (`polyphase`, as hoplite does)
   - **cut it into 5-second windows** (Perch's fixed input size)
   - **peak-normalize each window** — hoplite's default preprocessing
   - **run the model** to get a score for every species
   - turn scores into **probabilities** (softmax) and keep the **top-k per window**.
4. **Filter** by a confidence threshold (one value, or a whole range) and **save**
   the detections as CSV.
5. Optionally **cut out audio clips** of the detections to listen to.

### Considerations

- Perch V2 is **not** amplitude-invariant, so **two preprocessing choices move the scores**. Both are defaults of `perch-hoplite` (the official Perch library), and — worth knowing — **neither is documented in the Perch 2.0 paper**, whose Appendix A.2 otherwise specifies the frontend down to the log floor. So "hoplite's default" is the strongest claim available; "Perch's official preprocessing" is not.
  - **Peak normalization** (`TARGET_PEAK = 0.25`) — each 5-second window has its DC offset removed and its peak scaled to 0.25. Feed raw audio and you still get *an* answer, just a different one.
  - **Resampler** (`RESAMPLER = "polyphase"`) — hoplite resamples with `polyphase`; `librosa.load`'s own default is `soxr_hq`. This only matters when the source is not already 32 kHz.
- **How much do they move things?** Measured here on a real 24 kHz recording (60 windows, top-1 species per window), against the real model:

  | stage | mean Δ | mean \|Δ\| | max \|Δ\| |
  |---|---|---|---|
  | normalization (under `soxr_hq`) | −0.0197 | 0.0248 | 0.0836 |
  | normalization (under `polyphase`) | −0.0336 | 0.0384 | 0.1010 |
  | resampler (un-normalized) | −0.0127 | 0.0284 | 0.1182 |
  | resampler (normalized) | −0.0265 | 0.0355 | 0.1695 |

  Two things follow. **Neither stage dominates** — they are the same size, and each one's effect *depends on the other*, so blaming a discrepancy with another tool on one of them alone is wrong. And **normalizing mostly *lowers* the score** (49/60 and 52/60 windows), raising it only where the window's peak already exceeded 0.25 — so "skipping normalization gives low scores" is backwards.
- The model's window is pinned at **5 s** (the signature is `(None, 160000)`); only the *hop* between windows is tunable.
- **Verified against `perch-hoplite` itself**, not just described: on a real 24 kHz recording,
  the framing + normalization above are **bit-identical** to hoplite's `frame_audio` +
  `normalize_audio`, and the resulting softmax matches `TaxonomyModelTF.embed` to
  **2.4e-06** (float32 rounding — four orders of magnitude below the effects in the table),
  with the same top-1 species in 60/60 windows and an identical 14,795-class list. So this
  notebook reproduces the official library rather than approximating it. The only residual
  difference is that hoplite resamples in float64 and casts down, while `librosa.load`
  works in float32.
- Note for **stereo** input: `librosa.load(mono=True)` *averages* the channels, whereas
  hoplite keeps only the **first** channel. Mono files (like these) are unaffected.
- If you are comparing against another tool, match **both** knobs before concluding anything.

## Before you run

This project (**Perch_notebooks**) already contains everything this notebook needs.

### 1 · The environment

The Python environment lives in `.venv/`, built with [uv](https://docs.astral.sh/uv/)
from `pyproject.toml`: **Python 3.11** with TensorFlow 2.21, librosa, kagglehub,
soundfile, numpy, pandas and Jupyter. It is already installed — to rebuild it from
scratch:

```bash
cd ~/projects/Perch_notebooks
uv sync
```

Pick the right kernel. Top-right of this notebook → *Select Kernel* →
**Perch Notebooks (3.11)** (or `.venv/bin/python`). If the import cell below raises
`ModuleNotFoundError: No module named 'tensorflow'`, you are on the wrong kernel — that
is the single most common cause.

### 2 · Audio codecs (optional)

`.wav`, `.flac` and `.ogg` work out of the box: `libsndfile` ships inside the
`soundfile` wheel. Only `.mp3` / `.m4a` need a system decoder:

```bash
sudo apt-get install -y ffmpeg
```

### 3 · A Kaggle account

Perch V2 is downloaded from Kaggle on the first run, cached under
`~/.cache/kagglehub` afterwards, so it only happens once.

1. Accept the model's terms once on the
   [model page](https://www.kaggle.com/models/google/bird-vocalization-classifier).
2. Create a token: Kaggle → *Settings* → *API* → *Create New Token*.
3. Export it **before** launching VS Code / Jupyter, so the kernel inherits it:

```bash
export KAGGLE_API_TOKEN=KGAT_xxxxxxxxxxxxxxxxxxxxxxxx
```

A classic `~/.kaggle/kaggle.json`, or `KAGGLE_USERNAME` + `KAGGLE_KEY`, works too.
(Put the `export` in `~/.bashrc` to make it stick across shells.)

### 4 · Your recordings

Drop audio into **`data/recordings/`**, or point `INPUT_DIR` (§5) at any folder you
like. Results are written to `data/outputs/`.

## 0 · Libraries

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow's start-up logs

import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import tensorflow as tf

print("TensorFlow", tf.__version__, "— libraries ready.")

## 1 · Download Perch V2 from Kaggle

`kagglehub` fetches the model the **first** time and caches it under
`~/.cache/kagglehub` for every run afterwards. The first download needs a Kaggle
API token in your environment (create one at Kaggle → *Settings* → *API*, then
`export KAGGLE_API_TOKEN=...` before launching Jupyter).

We use the **`perch_v2_cpu`** build, which runs on any machine. (The plain
`perch_v2` build is compiled for CUDA and will not run on a CPU.)


In [ ]:
import kagglehub

MODEL_SLUG = "google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu"

model_path = Path(kagglehub.model_download(MODEL_SLUG))
print("Model files at:", model_path)

## 2 · Load the model and the species labels

A Perch SavedModel is called through its `serving_default` signature: you hand it a
batch of 5-second windows shaped `[N, 160000]` (float32) and it returns, among
other things, a `label` array of shape `[N, 14795]` — one raw score (*logit*) per
species, per window.

The species names live in `assets/labels.csv` inside the downloaded model. Its
first line is a namespace tag, so we skip it; the remaining 14,795 names line up
one-to-one with the `label` scores.


In [ ]:
model = tf.saved_model.load(str(model_path))
infer = model.signatures["serving_default"]   # inputs=[N, 160000] float32  ->  {"label", "embedding", ...}

SAMPLE_RATE    = 32_000                        # Perch's required input rate (Hz)
WINDOW_S       = 5.0                           # Perch's fixed window; do NOT change
WINDOW_SAMPLES = int(WINDOW_S * SAMPLE_RATE)   # 160000 samples per window

# The two preprocessing knobs. Both are perch-hoplite's defaults (see the notes at the
# top): they are choices, not fixed requirements, and they are why two tools running
# "the same" model can disagree on confidence. Change them only to reproduce another
# pipeline, and record what you used.
RESAMPLER   = "polyphase"   # hoplite's resampler. librosa.load's own default is "soxr_hq"
TARGET_PEAK = 0.25          # hoplite's per-window peak normalization. None disables it

label_lines = (model_path / "assets" / "labels.csv").read_text().splitlines()
CLASS_NAMES = [ln.strip() for ln in label_lines[1:] if ln.strip()]   # skip the header line
print(f"Loaded {len(CLASS_NAMES)} species classes. First few: {CLASS_NAMES[:3]}")

## 3 · Preprocessing

Three small functions:

- **`load_audio`** — read any common audio file as mono and resample to 32 kHz with
  `RESAMPLER`. Files already at 32 kHz are unaffected by the choice.
- **`frame_windows`** — slice the recording into 5-second windows, stepping forward
  by `hop_s` seconds each time. Windows shorter than 5 s (a very short file) are
  padded; a leftover tail shorter than one window is dropped.
- **`peak_normalize`** — for each window, subtract the mean (remove DC offset) and
  scale so the loudest sample sits at `TARGET_PEAK`. Setting `TARGET_PEAK = None`
  skips it (like hoplite); setting it to `0` would multiply the audio by zero and
  feed the model **silence** — it would not fail, it would just score nothing.


In [ ]:
def load_audio(path):
    """Load an audio file as mono float32 at Perch's 32 kHz rate.

    RESAMPLER is passed explicitly: librosa.load defaults to "soxr_hq", while
    perch-hoplite resamples with "polyphase". On 24 kHz input the two disagree by
    ~0.03 mean |delta| on the top confidence, so leaving it implicit silently
    decides whether this notebook reproduces hoplite or not.
    """
    audio, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True, res_type=RESAMPLER)
    return audio.astype(np.float32)


def frame_windows(audio, hop_s):
    """Cut audio into Perch's 5 s windows, stepping by hop_s seconds.

    Returns an array of shape [n_windows, 160000].
    """
    hop_samples = int(hop_s * SAMPLE_RATE)
    if len(audio) < WINDOW_SAMPLES:
        audio = librosa.util.pad_center(audio, size=WINDOW_SAMPLES, axis=-1)
    frames = librosa.util.frame(
        audio, frame_length=WINDOW_SAMPLES, hop_length=hop_samples, axis=-1
    ).swapaxes(-1, -2)
    return frames


def peak_normalize(frames, target_peak=TARGET_PEAK):
    """Per-window preprocessing: remove DC offset, scale the peak to target_peak.

    This mirrors perch-hoplite's normalize_audio with its default target_peak=0.25.
    Perch V2 is NOT amplitude-invariant, so this shifts the scores — measured on a
    24 kHz recording it *lowers* the top confidence in ~82% of windows and raises it
    where the window peak already exceeded 0.25 (mean |delta| ~0.03).

    target_peak=None returns the audio untouched (hoplite's way of disabling this).
    Silent windows (peak 0) are left at zero.
    """
    if target_peak is None:
        return frames.astype(np.float32)
    frames = frames.astype(np.float32).copy()
    frames -= frames.mean(axis=-1, keepdims=True)
    peak = np.max(np.abs(frames), axis=-1, keepdims=True)
    frames = np.divide(frames, peak, out=np.zeros_like(frames), where=peak > 0.0)
    return frames * target_peak

## 4 · Inference 

`window_probabilities` runs the model on a batch of windows and converts the raw
scores into **probabilities** with softmax.

`identify_file` ties it together for one recording and returns a table with the
top-k species for every window. 

In [ ]:
def window_probabilities(frames):
    """Run Perch on a batch of windows -> softmax probabilities [n_windows, n_classes]."""
    logits = infer(inputs=tf.constant(frames, dtype=tf.float32))["label"].numpy()
    return tf.nn.softmax(logits, axis=-1).numpy()


def identify_file(path, hop_s, top_k):
    """Full per-file pipeline -> DataFrame of the top-k predictions per window."""
    audio = load_audio(path)
    frames = frame_windows(audio, hop_s)
    frames = peak_normalize(frames)             # <-- hoplite's default, BEFORE inference
    probs = window_probabilities(frames)        # [n_windows, n_classes]

    rows = []
    for w, prob in enumerate(probs):
        start_s = w * hop_s
        top_idx = np.argsort(prob)[::-1][:top_k]
        for rank, idx in enumerate(top_idx, start=1):
            rows.append({
                "file": str(path),
                "start_s": round(start_s, 3),
                "end_s": round(start_s + WINDOW_S, 3),
                "window_s": WINDOW_S,
                "hop_s": hop_s,
                "rank": rank,
                "label": CLASS_NAMES[idx],
                "confidence": float(prob[idx]),
            })
    return pd.DataFrame(rows)

## 5 · Choose your input & output folders

- **`INPUT_DIR`** — a folder of recordings (searched recursively; `.wav .flac .ogg
  .mp3 .m4a .aiff .aif` are accepted).
- **`OUTPUT_DIR`** — where results are written (created automatically).


In [ ]:
# Anchor paths to the project folder (the one containing `data/`), walking up from
# wherever Jupyter was launched. VS Code starts the kernel in the notebook's own
# folder, a terminal `jupyter lab` starts it wherever you ran it — this works for both.
PROJECT_DIR = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir()),
    Path.cwd(),
)

INPUT_DIR  = PROJECT_DIR / "data" / "recordings"   # <-- change me: your recordings folder
OUTPUT_DIR = PROJECT_DIR / "data" / "outputs"      # <-- change me (optional): where results go

print("Project folder:", PROJECT_DIR)

AUDIO_EXTS = {".wav", ".flac", ".ogg", ".mp3", ".m4a", ".aiff", ".aif"}

def discover_audio(root):
    """Return sorted audio files anywhere under `root`."""
    return sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in AUDIO_EXTS)

Confirm the files are found before running the model.


In [ ]:
if not INPUT_DIR.exists():
    print(f"⚠ Input folder not found: {INPUT_DIR.resolve()}")
    print("  Point INPUT_DIR at a real folder of recordings and re-run this cell.")
else:
    found = discover_audio(INPUT_DIR)
    print(f"Found {len(found)} audio file(s) under {INPUT_DIR.resolve()}")
    for f in found[:10]:
        print("  -", f.name)

## 6 · Hop size and Top-K species

- **`HOP_S`** — how far the 5-second window slides each step. `5.0` gives
  back-to-back windows, while a smaller value overlaps them.
- **`TOP_K`** — how many of the highest-scoring species to keep per window.

In [ ]:
HOP_S = 5.0   # step between 5 s windows (s). < 5.0 overlaps them
TOP_K = 3     # keep this many highest-scoring species per window

## 7 · Confidence threshold(s)

A prediction becomes a *detection* when its confidence is **≥** the threshold.

A **sweep** can be triggered True/False in order to test multiple thresholds, getting one output folder for each one.

In [ ]:
RUN_SWEEP = False    # False = one threshold; True = a whole range

THRESHOLD = 0.1                              # used when RUN_SWEEP is False
SWEEP_START, SWEEP_END, SWEEP_STEP = 0.1, 1.0, 0.1   # used when RUN_SWEEP is True

def threshold_values():
    """The list of thresholds to export at."""
    if not RUN_SWEEP:
        return [THRESHOLD]
    values, v = [], SWEEP_START
    while v <= SWEEP_END + 1e-9:
        values.append(round(v, 6))
        v += SWEEP_STEP
    return values

## 8 · Extract audio segments (optional)

To verify detections, we can cut a short clip around each recording. Rather than
export *every* detection, we sample a balanced spread across confidence: the 0–1
confidence range is split into equal-width bands named **bins** (`BIN_WIDTH`), and at most
`MAX_PER_BIN` clips are kept **per species and per band**. Clips are saved into
one subfolder per species.

- **`CLIP_DURATION_S`** — length of the central clip.
- **`CONTEXT_S`** — extra seconds added on **each** side. a 5 s clip
  with `CONTEXT_S = 1.0` becomes a 7 s clip (1 s before + 5 s + 1 s after).
- **`BIN_WIDTH`** — width of each confidence band (0.1 → bands 0.0–0.1, 0.1–0.2, …).
- **`MAX_PER_BIN`** — max clips kept per species, per band.
- **`SEGMENT_SEED`** — makes the random sampling reproducible.


In [ ]:
EXTRACT_SEGMENTS = False   # set True to also export audio clips of detections

CLIP_DURATION_S = 5.0   # length of the central clip (s)
CONTEXT_S       = 0.0   # extra seconds on EACH side ("wings")
BIN_WIDTH       = 0.1   # width of each confidence band (0.1 -> 0.0-0.1, 0.1-0.2, ...)
MAX_PER_BIN     = 20    # max clips kept per species, per confidence band
SEGMENT_SEED    = 0     # random seed for the sampling


def extract_segments(detections, out_dir):
    """Sample detections evenly across confidence, separately for each species.

    Each detection is placed in a confidence band (its score // BIN_WIDTH). We then
    group by (species, band) and keep at most MAX_PER_BIN clips from each group.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(SEGMENT_SEED)
    total_s = CLIP_DURATION_S + 2.0 * max(0.0, CONTEXT_S)
    target = int(round(total_s * SAMPLE_RATE))

    dets = detections.copy()
    dets["_bin"] = (dets["confidence"] / BIN_WIDTH).astype(int)   # confidence band index
    chosen = []
    for _, grp in dets.groupby(["label", "_bin"]):                # per species, per band
        idx = list(grp.index)
        chosen += rng.sample(idx, MAX_PER_BIN) if len(idx) > MAX_PER_BIN else idx

    written = 0
    for i in chosen:
        row = dets.loc[i]
        center = 0.5 * (row["start_s"] + row["end_s"])
        start = max(0.0, center - total_s / 2.0)
        clip, _ = librosa.load(
            str(row["file"]), sr=SAMPLE_RATE, mono=True, offset=start, duration=total_s
        )
        clip = clip.astype(np.float32)
        if len(clip) < target:
            clip = np.pad(clip, (0, target - len(clip)))
        clip = clip[:target]

        species = re.sub(r"[^\w.-]+", "_", str(row["label"])).strip("_") or "unknown"
        species_dir = out_dir / species
        species_dir.mkdir(parents=True, exist_ok=True)
        name = f'{Path(row["file"]).stem}_{species}_{row["confidence"]:.3f}_{int(row["start_s"])}s.wav'
        sf.write(species_dir / name, clip, SAMPLE_RATE)
        written += 1
    return written

## 9 · Run the model over every recording

Loads and scores each file once. Progress prints per file. The result, `predictions`, holds the top-k species for every window of every recording, before any threshold is applied.


In [ ]:
files = discover_audio(INPUT_DIR)
if not files:
    raise SystemExit(f"No audio files found under {INPUT_DIR}")

per_file = []
for i, path in enumerate(files, start=1):
    print(f"[{i}/{len(files)}] {path.name}")
    per_file.append(identify_file(path, HOP_S, TOP_K))

predictions = pd.concat(per_file, ignore_index=True)
print(f"Done: {len(predictions)} window-predictions from {len(files)} file(s).")

## 10 · Apply the threshold(s) and save the output

For each threshold, we keep the predictions at or above it. Also creates a
`threshold_<value>/` folder: one CSV per recording, a combined `all_detections.csv`,
and a `summary.txt` of per-species counts. Also a `manifest.json` records the settings.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
counts_by_threshold = {}

for t in threshold_values():
    kept = predictions[predictions["confidence"] >= t].copy()
    kept["threshold"] = t
    kept = kept[[c for c in kept.columns if c != "file"] + ["file"]]   # file path as the last column

    thr_dir = OUTPUT_DIR / f"threshold_{t:.2f}"
    thr_dir.mkdir(parents=True, exist_ok=True)
    for fname, grp in kept.groupby("file"):
        grp.to_csv(thr_dir / f"{Path(fname).stem}.csv", index=False)
    kept.to_csv(thr_dir / "all_detections.csv", index=False)
    (thr_dir / "summary.txt").write_text(
        f"Threshold {t:.2f}\nDetections: {len(kept)}\n\n"
        + (kept["label"].value_counts().to_string() if len(kept) else "(none)"),
        encoding="utf-8",
    )
    counts_by_threshold[f"{t:.2f}"] = len(kept)

manifest = {
    "model_slug": MODEL_SLUG,
    "sample_rate": SAMPLE_RATE,
    "resampler": RESAMPLER,
    "target_peak": TARGET_PEAK,
    "window_s": WINDOW_S,
    "hop_s": HOP_S,
    "top_k": TOP_K,
    "thresholds": threshold_values(),
    "n_classes": len(CLASS_NAMES),
    "n_files": len(files),
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Detections per threshold:", counts_by_threshold)

### Extract the clips (if enabled)


In [ ]:
if EXTRACT_SEGMENTS:
    primary = threshold_values()[0]
    dets = pd.read_csv(OUTPUT_DIR / f"threshold_{primary:.2f}" / "all_detections.csv")
    n = extract_segments(dets, OUTPUT_DIR / f"threshold_{primary:.2f}" / "segments")
    print(f"Wrote {n} clip(s) under threshold_{primary:.2f}/segments/")
else:
    print("Segment extraction is off (EXTRACT_SEGMENTS = False).")

## 11 · Explore the results

Load the combined table for the first threshold, preview it, and count species.


In [ ]:
primary = threshold_values()[0]
agg_csv = OUTPUT_DIR / f"threshold_{primary:.2f}" / "all_detections.csv"
detections_df = pd.read_csv(agg_csv)
print(f"{len(detections_df)} detections at threshold {primary:.2f}  ->  {agg_csv}")
detections_df.head(15)

In [ ]:
if len(detections_df):
    print(detections_df["label"].value_counts().head(20))
else:
    print("No detections at this threshold — try a lower one.")

## 12 · Next: how high should the threshold be?

The confidence cutoff used above is a blunt instrument — the right value is
species-specific, and a 0.3 for one bird can be as reliable as a 0.7 for another.

To estimate it properly, sort the clips from §8 into `correct/` and `incorrect/`
folders and fit a logistic regression per species, solving for the confidence at which
a detection has a 95 % probability of being correct. That analysis lives in its own
notebook (`04_optimal_threshold_guided.ipynb` in the PerchLab repo) — we can bring it
into this project when you want it.